In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import numpy as np
from torchvision.transforms.v2 import RandAugment
import torch
from tqdm.notebook import tqdm
from torch.utils.tensorboard import SummaryWriter
import torch.nn as nn

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("../models"))

from dataset import get_data_loaders
from backbone import BackBone
from multiheadmodel import MultiHeadModel
from utils import deterministic, train, accuracy

/home/alumno1/miniconda3/envs/vision/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
backbone= BackBone()
backbone.load_state_dict(torch.load("../models/weights/backbone.pth"))
model = MultiHeadModel(backbone)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
for param in model.backbone.parameters(): 
    param.requires_grad = False

/home/alumno1/miniconda3/envs/vision/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/alumno1/miniconda3/envs/vision/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Comentario de claude para acelerar el entrenamiento de las cabezas:

Si el backbone está congelado, podés pre-calcular los embeddings una sola vez y entrenar solo sobre ellos — mucho más rápido

In [4]:
dataloaders = get_data_loaders(batch_size=512)

/home/alumno1/miniconda3/envs/vision/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Task [0, 1]: Train=9000, Val=1000, Test=2000
Task [2, 3]: Train=9000, Val=1000, Test=2000
Task [4, 5]: Train=9000, Val=1000, Test=2000
Task [6, 7]: Train=9000, Val=1000, Test=2000
Task [8, 9]: Train=9000, Val=1000, Test=2000


#### Fine Tuning Naive Task-IL

In [5]:
for i in range(5):
    model.add_head(i, 2)
    model.to(device)
    train_data = dataloaders[i][0]
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    epochs = 20

    train(model, train_data, optimizer, criterion, f"naive_task{i}", epochs, task_number=i, save=False)
    
    eval_data = dataloaders[i][1]
    acc = accuracy(model, eval_data, task_number=i)
    print(f"Task {i} Accuracy: {acc:.4f}")

model.save(f"../models/weights/naive_Task-IL.pth")

Evaluating: 100%|██████████| 2/2 [00:00<00:00, 11.95batch/s]


Task 0 Accuracy: 0.9890


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 12.30batch/s]


Task 1 Accuracy: 0.6080


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 10.38batch/s]


Task 2 Accuracy: 0.5530


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 12.02batch/s]


Task 3 Accuracy: 0.5240


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 12.54batch/s]


Task 4 Accuracy: 0.7720


#### Fine Tuning Naive Class-IL